In [3]:
#!pip install mlxtend

In [4]:
import pandas as pd

#### Create the dataset

In [5]:
dataset = [
    ['Apple', 'Beer', 'Rice', 'Chicken'],
    ['Apple', 'Beer', 'Rice'],
    ['Apple', 'Beer'],
    ['Apple', 'Pear'],
    ['Milk', 'Beer', 'Rice', 'Chicken'],
    ['Milk', 'Beer', 'Rice'],
    ['Milk', 'Beer'],
    ['Milk', 'Pear']
]

In [6]:
dataset

[['Apple', 'Beer', 'Rice', 'Chicken'],
 ['Apple', 'Beer', 'Rice'],
 ['Apple', 'Beer'],
 ['Apple', 'Pear'],
 ['Milk', 'Beer', 'Rice', 'Chicken'],
 ['Milk', 'Beer', 'Rice'],
 ['Milk', 'Beer'],
 ['Milk', 'Pear']]

#### Transaction encoder for creating structured data

In [7]:
from mlxtend.preprocessing import TransactionEncoder

In [8]:
te = TransactionEncoder()

In [9]:
df_enc = te.fit_transform(dataset)

In [10]:
df_enc

array([[ True,  True,  True, False, False,  True],
       [ True,  True, False, False, False,  True],
       [ True,  True, False, False, False, False],
       [ True, False, False, False,  True, False],
       [False,  True,  True,  True, False,  True],
       [False,  True, False,  True, False,  True],
       [False,  True, False,  True, False, False],
       [False, False, False,  True,  True, False]])

In [11]:
te.columns_

['Apple', 'Beer', 'Chicken', 'Milk', 'Pear', 'Rice']

In [12]:
df = pd.DataFrame(df_enc, columns= te.columns_)

In [13]:
df

,Apple,Beer,Chicken,Milk,Pear,Rice
0,True,True,True,False,False,True
1,True,True,False,False,False,True
2,True,True,False,False,False,False
3,True,False,False,False,True,False
4,False,True,True,True,False,True
5,False,True,False,True,False,True
6,False,True,False,True,False,False
7,False,False,False,True,True,False


#### step 1: Generate the frequent itemsets

In [14]:
from mlxtend.frequent_patterns import apriori

In [15]:
freq_itemsets = apriori(df, min_support= 0.25, use_colnames= True)

In [16]:
freq_itemsets

,support,itemsets
0,0.500,(Apple)
1,0.750,(Beer)
2,0.250,(Chicken)
3,0.500,(Milk)
4,0.250,(Pear)
5,0.500,(Rice)
6,0.375,"(Apple, Beer)"
7,0.250,"(Rice, Apple)"
8,0.250,"(Chicken, Beer)"
9,0.375,"(Milk, Beer)"


#### Generate the association rules

In [17]:
from mlxtend.frequent_patterns import association_rules

In [18]:
rules = association_rules(freq_itemsets, metric= 'confidence',
                         min_threshold= 0.50)

In [19]:
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
0,(Apple),(Beer),0.500,0.750,0.375,0.750000,1.000000,0.0000,1.00,0.000000
1,(Beer),(Apple),0.750,0.500,0.375,0.500000,1.000000,0.0000,1.00,0.000000
2,(Rice),(Apple),0.500,0.500,0.250,0.500000,1.000000,0.0000,1.00,0.000000
3,(Apple),(Rice),0.500,0.500,0.250,0.500000,1.000000,0.0000,1.00,0.000000
4,(Chicken),(Beer),0.250,0.750,0.250,1.000000,1.333333,0.0625,inf,0.333333
5,(Milk),(Beer),0.500,0.750,0.375,0.750000,1.000000,0.0000,1.00,0.000000
6,(Beer),(Milk),0.750,0.500,0.375,0.500000,1.000000,0.0000,1.00,0.000000
7,(Rice),(Beer),0.500,0.750,0.500,1.000000,1.333333,0.1250,inf,0.500000
8,(Beer),(Rice),0.750,0.500,0.500,0.666667,1.333333,0.1250,1.50,1.000000
9,(Rice),(Chicken),0.500,0.250,0.250,0.500000,2.000000,0.1250,1.50,1.000000


In [20]:
rules.columns

Index(['antecedents', 'consequents', 'antecedent support',
       'consequent support', 'support', 'confidence', 'lift', 'leverage',
       'conviction', 'zhangs_metric'],
      dtype='object')

### Extract specific cols

In [21]:
rules = rules[['antecedents', 'consequents','support', 'confidence']]

In [22]:
rules

,antecedents,consequents,support,confidence
0,(Apple),(Beer),0.375,0.750000
1,(Beer),(Apple),0.375,0.500000
2,(Rice),(Apple),0.250,0.500000
3,(Apple),(Rice),0.250,0.500000
4,(Chicken),(Beer),0.250,1.000000
5,(Milk),(Beer),0.375,0.750000
6,(Beer),(Milk),0.375,0.500000
7,(Rice),(Beer),0.500,1.000000
8,(Beer),(Rice),0.500,0.666667
9,(Rice),(Chicken),0.250,0.500000


### Extract using condition

In [23]:
rules[rules['confidence'] > 0.5]

,antecedents,consequents,support,confidence
0,(Apple),(Beer),0.375,0.750000
4,(Chicken),(Beer),0.250,1.000000
5,(Milk),(Beer),0.375,0.750000
7,(Rice),(Beer),0.500,1.000000
8,(Beer),(Rice),0.500,0.666667
10,(Chicken),(Rice),0.250,1.000000
13,"(Rice, Apple)",(Beer),0.250,1.000000
15,"(Apple, Beer)",(Rice),0.250,0.666667
18,"(Rice, Chicken)",(Beer),0.250,1.000000
20,"(Chicken, Beer)",(Rice),0.250,1.000000


In [24]:
rules[(rules['confidence'] > 0.5) & (rules['support'] > 0.25)]

,antecedents,consequents,support,confidence
0,(Apple),(Beer),0.375,0.750000
5,(Milk),(Beer),0.375,0.750000
7,(Rice),(Beer),0.500,1.000000
8,(Beer),(Rice),0.500,0.666667


### Recommendation

In [25]:
rules[rules['antecedents'] == {'Rice'}]

,antecedents,consequents,support,confidence
2,(Rice),(Apple),0.25,0.5
7,(Rice),(Beer),0.50,1.0
9,(Rice),(Chicken),0.25,0.5
11,(Rice),(Milk),0.25,0.5
16,(Rice),"(Apple, Beer)",0.25,0.5
21,(Rice),"(Chicken, Beer)",0.25,0.5
26,(Rice),"(Milk, Beer)",0.25,0.5


In [26]:
rules[(rules['antecedents'] == {'Rice'}) & (rules['confidence'] > 0.5)]

,antecedents,consequents,support,confidence
7,(Rice),(Beer),0.5,1.0


In [30]:
import warnings
warnings.filterwarnings('ignore')

### sort values

In [31]:
rules.sort_values(by = 'confidence', ascending= False, inplace= True)

In [32]:
rules

,antecedents,consequents,support,confidence
22,(Chicken),"(Rice, Beer)",0.250,1.000000
4,(Chicken),(Beer),0.250,1.000000
13,"(Rice, Apple)",(Beer),0.250,1.000000
7,(Rice),(Beer),0.500,1.000000
18,"(Rice, Chicken)",(Beer),0.250,1.000000
23,"(Rice, Milk)",(Beer),0.250,1.000000
10,(Chicken),(Rice),0.250,1.000000
20,"(Chicken, Beer)",(Rice),0.250,1.000000
0,(Apple),(Beer),0.375,0.750000
5,(Milk),(Beer),0.375,0.750000


### save the rules

In [33]:
rules.to_csv('rules.csv', index= False)